# Deep Agents: Building Complex Agents for Long-Horizon Tasks

In this notebook, we'll explore **Deep Agents** - a new approach to building AI agents that can handle complex, multi-step tasks over extended periods. We'll implement all four key elements of Deep Agents while building on our Personal Wellness Assistant use case.

**Learning Objectives:**
- Understand the four key elements of Deep Agents: Planning, Context Management, Subagent Spawning, and Long-term Memory
- Implement each element progressively using the `deepagents` package
- Learn to use Skills for progressive capability disclosure
- Use the `deepagents-cli` for interactive agent sessions

## Table of Contents:

- **Breakout Room #1:** Deep Agent Foundations
  - Task 1: Dependencies & Setup
  - Task 2: Understanding Deep Agents
  - Task 3: Planning with Todo Lists
  - Task 4: Context Management with File Systems
  - Task 5: Basic Deep Agent
  - Question #1 & Question #2
  - Activity #1: Build a Research Agent

- **Breakout Room #2:** Advanced Features & Integration
  - Task 6: Subagent Spawning
  - Task 7: Long-term Memory Integration
  - Task 8: Skills - On-Demand Capabilities
  - Task 9: Using deepagents-cli
  - Task 10: Building a Complete Deep Agent System
  - Question #3 & Question #4
  - Activity #2: Build a Wellness Coach Agent

---
# 🤝 Breakout Room #1
## Deep Agent Foundations

## Task 1: Dependencies & Setup

Before we begin, make sure you have:

1. **API Keys** for:
   - Anthropic (default for Deep Agents) or OpenAI
   - LangSmith (optional, for tracing)
   - Tavily (optional, for web search)

2. **Dependencies installed** via `uv sync`

3. **For the CLI** (Task 9): `uv pip install deepagents-cli`

### Environment Setup

You can either:
- Create a `.env` file with your API keys (recommended):
  ```
  ANTHROPIC_API_KEY=your_key_here
  OPENAI_API_KEY=your_key_here
  LANGCHAIN_API_KEY=your_key_here
  ```
- Or enter them interactively when prompted

In [1]:
# Core imports
import os
import getpass
from uuid import uuid4
from typing import Annotated, TypedDict, Literal

import nest_asyncio
nest_asyncio.apply()  # Required for async operations in Jupyter

# Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

def get_api_key(env_var: str, prompt: str) -> str:
    """Get API key from environment or prompt user."""
    value = os.environ.get(env_var, "")
    if not value:
        value = getpass.getpass(prompt)
        if value:
            os.environ[env_var] = value
    return value

In [2]:
# Set Anthropic API Key (default for Deep Agents)
anthropic_key = get_api_key("ANTHROPIC_API_KEY", "Anthropic API Key: ")
if anthropic_key:
    print("Anthropic API key set")
else:
    print("Warning: No Anthropic API key configured")

Anthropic API key set


In [3]:
# Optional: OpenAI for alternative models and subagents
openai_key = get_api_key("OPENAI_API_KEY", "OpenAI API Key (press Enter to skip): ")
if openai_key:
    print("OpenAI API key set")
else:
    print("OpenAI API key not configured (optional)")

OpenAI API key set


In [4]:
# Optional: LangSmith for tracing
langsmith_key = get_api_key("LANGCHAIN_API_KEY", "LangSmith API Key (press Enter to skip): ")

if langsmith_key:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = f"AIE9 - Deep Agents - {uuid4().hex[0:8]}"
    print(f"LangSmith tracing enabled. Project: {os.environ['LANGCHAIN_PROJECT']}")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("LangSmith tracing disabled")

LangSmith tracing enabled. Project: AIE9 - Deep Agents - f50c19b8


In [5]:
# Verify deepagents installation
from deepagents import create_deep_agent
print("deepagents package imported successfully!")

# Test with a simple agent
test_agent = create_deep_agent()
result = test_agent.invoke({
    "messages": [{"role": "user", "content": "Say 'Deep Agents ready!' in exactly those words."}]
})
print(result["messages"][-1].content)

deepagents package imported successfully!
Deep Agents ready!


## Task 2: Understanding Deep Agents

**Deep Agents** represent a shift from simple tool-calling loops to sophisticated agents that can handle complex, long-horizon tasks. They address four key challenges:

### The Four Key Elements

| Element | Challenge Addressed | Implementation |
|---------|---------------------|----------------|
| **Planning** | "What should I do?" | Todo lists that persist task state |
| **Context Management** | "What do I know?" | File systems for storing/retrieving info |
| **Subagent Spawning** | "Who can help?" | Task tool for delegating to specialists |
| **Long-term Memory** | "What did I learn?" | LangGraph Store for cross-session memory |

### Deep Agents vs Traditional Agents

```
Traditional Agent Loop:
┌─────────────────────────────────────┐
│  User Query                         │
│       ↓                             │
│  Think → Act → Observe → Repeat     │
│       ↓                             │
│  Response                           │
└─────────────────────────────────────┘
Problems: Context bloat, no delegation,
          loses track of complex tasks

Deep Agent Architecture:
┌─────────────────────────────────────────────────────────┐
│                    Deep Agent                           │
├─────────────────────────────────────────────────────────┤
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐   │
│  │   PLANNING   │  │   CONTEXT    │  │   MEMORY     │   │
│  │              │  │  MANAGEMENT  │  │              │   │
│  │ write_todos  │  │              │  │   Store      │   │
│  │ update_todo  │  │  read_file   │  │  namespace   │   │
│  │ list_todos   │  │  write_file  │  │  get/put     │   │
│  │              │  │  edit_file   │  │              │   │
│  └──────────────┘  │  ls          │  └──────────────┘   │
│                    └──────────────┘                     │
│  ┌──────────────────────────────────────────────────┐   │
│  │              SUBAGENT SPAWNING                   │   │
│  │                                                  │   │
│  │  task(prompt, tools, model, system_prompt)       │   │
│  │       ↓              ↓              ↓            │   │
│  │  ┌────────┐    ┌────────┐    ┌────────┐          │   │
│  │  │Research│    │Writing │    │Analysis│          │   │
│  │  │Subagent│    │Subagent│    │Subagent│          │   │
│  │  └────────┘    └────────┘    └────────┘          │   │
│  └──────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────┘
```

### When to Use Deep Agents

| Use Case | Traditional Agent | Deep Agent |
|----------|-------------------|------------|
| Simple Q&A | ✅ | Overkill |
| Single-step tool use | ✅ | Overkill |
| Multi-step research | ⚠️ May lose track | ✅ |
| Complex projects | ❌ Context overflow | ✅ |
| Parallel task execution | ❌ | ✅ |
| Long-running sessions | ❌ | ✅ |

### Key Insight: "Planning is Context Engineering"

Deep Agents treat planning not as a separate phase, but as **context engineering**:
- Todo lists aren't just task trackers—they're **persistent context** about what to do
- File systems aren't just storage—they're **extended memory** beyond the context window
- Subagents aren't just helpers—they're **context isolation** to prevent bloat

## Task 3: Planning with Todo Lists

The first key element of Deep Agents is **Planning**. Instead of trying to hold all task state in the conversation, Deep Agents use structured todo lists.

### Why Todo Lists?

1. **Persistence**: Tasks survive across conversation turns
2. **Visibility**: Both agent and user can see progress
3. **Structure**: Clear tracking of what's done vs pending
4. **Recovery**: Agent can resume from where it left off

### Todo List Tools

| Tool | Purpose |
|------|----------|
| `write_todos` | Create a structured task list |
| `update_todo` | Mark tasks as complete/in-progress |
| `list_todos` | View current task state |

In [8]:
from langchain_core.tools import tool
from typing import List, Optional
import json

# Simple in-memory todo storage for demonstration
# In production, Deep Agents use persistent storage
TODO_STORE = {}

@tool
def write_todos(todos: List[dict]) -> str:
    """Create a list of todos for tracking task progress.
    
    Args:
        todos: List of todo items, each with 'title' and optional 'description'
    
    Returns:
        Confirmation message with todo IDs
    """
    created = []
    for i, todo in enumerate(todos):
        todo_id = f"todo_{len(TODO_STORE) + i + 1}"
        TODO_STORE[todo_id] = {
            "id": todo_id,
            "title": todo.get("title", "Untitled"),
            "description": todo.get("description", ""),
            "status": "pending"
        }
        created.append(todo_id)
    return f"Created {len(created)} todos: {', '.join(created)}"

@tool
def update_todo(todo_id: str, status: Literal["pending", "in_progress", "completed"]) -> str:
    """Update the status of a todo item.
    
    Args:
        todo_id: The ID of the todo to update
        status: New status (pending, in_progress, completed)
    
    Returns:
        Confirmation message
    """
    if todo_id not in TODO_STORE:
        return f"Todo {todo_id} not found"
    TODO_STORE[todo_id]["status"] = status
    return f"Updated {todo_id} to {status}"

@tool
def list_todos() -> str:
    """List all todos with their current status.
    
    Returns:
        Formatted list of all todos
    """
    if not TODO_STORE:
        return "No todos found"
    
    result = []
    for todo_id, todo in TODO_STORE.items():
        status_emoji = {"pending": "⬜", "in_progress": "🔄", "completed": "✅"}
        emoji = status_emoji.get(todo["status"], "❓")
        result.append(f"{emoji} [{todo_id}] {todo['title']} ({todo['status']})")
    return "\n".join(result)

print("Todo tools defined!")

Todo tools defined!


In [9]:
# Test the todo tools
TODO_STORE.clear()  # Reset for demo

# Create some wellness todos
result = write_todos.invoke({
    "todos": [
        {"title": "Assess current sleep patterns", "description": "Review user's sleep schedule and quality"},
        {"title": "Research sleep improvement strategies", "description": "Find evidence-based techniques"},
        {"title": "Create personalized sleep plan", "description": "Combine findings into actionable steps"},
    ]
})
print(result)
print("\nCurrent todos:")
print(list_todos.invoke({}))

Created 3 todos: todo_1, todo_3, todo_5

Current todos:
⬜ [todo_1] Assess current sleep patterns (pending)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


In [10]:
# Simulate progress
update_todo.invoke({"todo_id": "todo_1", "status": "completed"})
update_todo.invoke({"todo_id": "todo_2", "status": "in_progress"})

print("After updates:")
print(list_todos.invoke({}))

After updates:
✅ [todo_1] Assess current sleep patterns (completed)
⬜ [todo_3] Research sleep improvement strategies (pending)
⬜ [todo_5] Create personalized sleep plan (pending)


## Task 4: Context Management with File Systems

The second key element is **Context Management**. Deep Agents use file systems to:

1. **Offload large content** - Store research, documents, and results to disk
2. **Persist across sessions** - Files survive beyond conversation context
3. **Share between subagents** - Subagents can read/write shared files
4. **Prevent context overflow** - Large tool results automatically saved to disk

### Automatic Context Management

Deep Agents automatically handle context limits:
- **Large result offloading**: Tool results >20k tokens → saved to disk
- **Proactive offloading**: At 85% context capacity → agent saves state to disk
- **Summarization**: Long conversations get summarized while preserving intent

### File System Tools

| Tool | Purpose |
|------|----------|
| `ls` | List directory contents |
| `read_file` | Read file contents |
| `write_file` | Create/overwrite files |
| `edit_file` | Make targeted edits |

In [12]:
import os
from pathlib import Path

# Create a workspace directory for our agent
WORKSPACE = Path("workspace")
WORKSPACE.mkdir(exist_ok=True)

@tool
def ls(path: str = ".") -> str:
    """List contents of a directory.
    
    Args:
        path: Directory path to list (default: current directory)
    
    Returns:
        List of files and directories
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"Directory not found: {path}"
    
    items = []
    for item in sorted(target.iterdir()):
        prefix = "[DIR]" if item.is_dir() else "[FILE]"
        size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
        items.append(f"{prefix} {item.name}{size}")
    
    return "\n".join(items) if items else "(empty directory)"

@tool
def read_file(path: str) -> str:
    """Read contents of a file.
    
    Args:
        path: Path to the file to read
    
    Returns:
        File contents
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    return target.read_text()

@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file (creates or overwrites).
    
    Args:
        path: Path to the file to write
        content: Content to write to the file
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
    return f"Wrote {len(content)} characters to {path}"

@tool
def edit_file(path: str, old_text: str, new_text: str) -> str:
    """Edit a file by replacing text.
    
    Args:
        path: Path to the file to edit
        old_text: Text to find and replace
        new_text: Replacement text
    
    Returns:
        Confirmation message
    """
    target = WORKSPACE / path
    if not target.exists():
        return f"File not found: {path}"
    
    content = target.read_text()
    if old_text not in content:
        return f"Text not found in {path}"
    
    new_content = content.replace(old_text, new_text, 1)
    target.write_text(new_content)
    return f"Updated {path}"

print("File system tools defined!")
print(f"Workspace: {WORKSPACE.absolute()}")

File system tools defined!
Workspace: /Users/nikos/n/rvm/AIE9/07_Deep_Agents/workspace


In [13]:
# Test the file system tools
print("Current workspace contents:")
print(ls.invoke({"path": "."}))

Current workspace contents:
(empty directory)


In [14]:
# Create a research notes file
notes = """# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations
"""

result = write_file.invoke({"path": "research/sleep_notes.md", "content": notes})
print(result)

# Verify it was created
print("\nResearch directory:")
print(ls.invoke({"path": "research"}))

Wrote 242 characters to research/sleep_notes.md

Research directory:
[FILE] sleep_notes.md (242 bytes)


In [15]:
# Read and edit the file
print("File contents:")
print(read_file.invoke({"path": "research/sleep_notes.md"}))

File contents:
# Sleep Research Notes

## Key Findings
- Adults need 7-9 hours of sleep
- Consistent sleep schedule is important
- Blue light affects melatonin production

## TODO
- [ ] Review individual user needs
- [ ] Create personalized recommendations



## Task 5: Basic Deep Agent

Now let's create a basic Deep Agent using the `deepagents` package. This combines:
- Planning (todo lists)
- Context management (file system)
- A capable LLM backbone

### Configuring the FilesystemBackend

Deep Agents come with **built-in file tools** (`ls`, `read_file`, `write_file`, `edit_file`). To control where files are stored, we configure a `FilesystemBackend`:

```python
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(
    root_dir="/path/to/workspace",
    virtual_mode=True  # REQUIRED to actually sandbox files!
)
```

**Critical: `virtual_mode=True`**
- Without `virtual_mode=True`, agents can still write anywhere on the filesystem!
- The `root_dir` alone does NOT restrict file access
- `virtual_mode=True` blocks paths with `..`, `~`, and absolute paths outside root

In [16]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Configure the filesystem backend to use our workspace directory
# IMPORTANT: virtual_mode=True is required to actually restrict paths to root_dir
# Without it, agents can still write anywhere on the filesystem!
workspace_path = Path("workspace").absolute()
filesystem_backend = FilesystemBackend(
    root_dir=str(workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Combine our custom tools (for todo tracking)
# Note: Deep Agents has built-in file tools (ls, read_file, write_file, edit_file)
# that will use the configured FilesystemBackend
custom_tools = [
    write_todos,
    update_todo,
    list_todos,
]

# Create a basic Deep Agent
wellness_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=custom_tools,
    backend=filesystem_backend,  # Configure where files are stored
    system_prompt="""You are a Personal Wellness Assistant that helps users improve their health.

When given a complex task:
1. First, create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Be thorough but concise. Always explain your reasoning."""
)

print(f"Basic Deep Agent created!")
print(f"File operations sandboxed to: {workspace_path}")

Basic Deep Agent created!
File operations sandboxed to: /Users/nikos/n/rvm/AIE9/07_Deep_Agents/workspace


In [17]:
# Reset todo store for fresh demo
TODO_STORE.clear()

# Test with a multi-step wellness task
result = wellness_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please create a personalized sleep improvement plan for me and save it to a file."""
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
## ✅ Your Personalized Sleep Improvement Plan is Complete!

I've created a comprehensive 4-week sleep transformation plan specifically tailored to your challenges. The plan is saved as `/personalized_sleep_improvement_plan.md` and addresses all three of your main issues:

### **🎯 Key Highlights:**

**Week 1 Focus** (Start immediately):
- **Fixed wake time every day** (most important change!)
- **Phone charging station outside bedroom** (start tonight!)
- **Morning light exposure** within 30 minutes of waking
- **1-hour screen curfew** before bedtime

**Expected Timeline:**
- **3-5 days:** Easier morning wake-ups
- **1-2 weeks:** Improved morning alertness  
- **2-4 weeks:** Consistent energy levels
- **5-8 weeks:** Complete circadian rhythm optimization

### **🚀 Start Tonight:**
The single most impactful change you can make **right now** is moving your phone charger out of your bedroom. This one action will immediately improve your sleep onset and quality.

### **📊 Why 

In [18]:
# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))

print("\n" + "="*50)
print("\nWorkspace contents:")
# List files in the workspace directory
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    else:
        print(f"  [DIR] {f.name}/")

Todo list after task:
✅ [todo_1] Analyze current sleep issues (completed)
✅ [todo_3] Research sleep improvement strategies (completed)
✅ [todo_5] Research sleep schedule consistency and circadian rhythm regulation (completed)
✅ [todo_7] Save plan to file (completed)
✅ [todo_6] Research digital device management and blue light exposure (completed)
✅ [todo_8] Research sleep environment optimization (completed)
✅ [todo_10] Research pre-sleep routine recommendations (completed)
✅ [todo_12] Research morning routine strategies for fatigue (completed)
✅ [todo_14] Research dietary and lifestyle factors (completed)
✅ [todo_16] Compile comprehensive structured summary (completed)


Workspace contents:
  [FILE] personalized_sleep_improvement_plan.md (6247 bytes)
  [DIR] research/
  [FILE] sleep_improvement_research_summary.md (7839 bytes)


---
## ❓ Question #1:

What are the **trade-offs** of using todo lists for planning? Consider:
- When might explicit planning overhead slow things down?
- How granular should todo items be?
- What happens if the agent creates todos but never completes them?

##### Answer:
###### Pros:
- The list is sequential and can be worked on in sequence always understanding and keep track of the progress in a logical fashion.
- The list helps break down a complex problem in smaller tasks.
- We can persist the list (and intermediate results) so we can resume the agent in case of an app failure without losing all the work (it's like autosave).
###### Cons:
- The agent might create a list that is either too high-level (which will lead to low quality) or low-level (which will lead to overkill).
- For a simple problem creating a list will consume more time and cost than necessary. The time of creating the list plus the time doing all the todos.

## ❓ Question #2:

How would you design a **context management strategy** for a wellness agent that:
- Needs to reference a large health document (16KB)
- Tracks user metrics over time
- Must remember user conditions (allergies, medications) for safety

What goes in files vs. in the prompt? What should never be offloaded?

##### Answer:
**Health document**: store it in a file and just reference it in the prompt. Don't put that in the prompt. Additionally we could chunk and store the doc in a semantic store for targeted retrieval, but that will depend on how the contents are organized. If the doc is not structured enough then maybe we just load all 16KB during a "research tool" step.

**User Metrics**: I would store those in a database as "memory" so we can run queries on it via a tool. Those will be updated over time so we would insert new records. If the metrics are simple enough we can just add entries in a file with timestamps to persist the entries like a simple appending log.

**User Conditions**: Those are part of a user profile so we should place it in a store as agent memory to be referenced across multiple threads. 

If by "offloaded" we mean "store outside the prompt" then we should never offload the actual instructions for that agent, especially the steps it needs to follow for checking the profile of the user for safety purposes. Those guardrails need to be in the system prompt.


---
## 🏗️ Activity #1: Build a Research Agent

Build a Deep Agent that can research a wellness topic and produce a structured report.

### Requirements:
1. Create todos for the research process
2. Read from the HealthWellnessGuide.txt in the data folder
3. Save findings to a structured markdown file
4. Update todo status as tasks complete

### Test prompt:
"Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."

In [20]:
### YOUR CODE HERE ###

# Step 1: Create a research agent with appropriate tools
# Hint: You'll need file tools to read the wellness guide
stress_workspace_path = Path("stress_workspace").absolute()
stress_backend = FilesystemBackend(
    root_dir=str(stress_workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

# Step 2: Add a tool to read from the data folder
# Hint: Use Path("data/HealthWellnessGuide.txt")
@tool
def read_health_wellness_guide() -> str:
    """Read the HealthWellnessGuide.txt file.

    Args:
        None

    Returns:
        The content of the HealthWellnessGuide.txt file
    """
    return Path("data/HealthWellnessGuide.txt").read_text()

stress_tools = [
    read_health_wellness_guide,
    write_todos,
    update_todo,
    list_todos,
]

# Step 3: Create the agent with a research-focused system prompt
stress_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=stress_tools,
    backend=stress_backend,
    system_prompt="""You are a health and wellness research specialist.
When given a complex task:
1. Create a todo list to track your progress
2. Work through each task, updating status as you go
3. Save important findings to files for reference
4. Provide a clear summary when complete

Use the HealthWellnessGuide.txt file exclusively to research the topic
Be thorough but concise. Always explain your reasoning."""
)

print(f"Stress Deep Agent created!")
print(f"File operations sandboxed to: {stress_workspace_path}")

# Step 4: Test with the stress management research task
TODO_STORE.clear()

result = stress_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Research stress management techniques and create a comprehensive guide with at least 5 evidence-based strategies."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

# Check what the agent created
print("Todo list after task:")
print(list_todos.invoke({}))    

Stress Deep Agent created!
File operations sandboxed to: /Users/nikos/n/rvm/AIE9/07_Deep_Agents/stress_workspace
Agent response:
Perfect! I have successfully completed all tasks. Let me provide you with a summary of the comprehensive stress management guide I've created.

## Summary

I've successfully researched and created a **Comprehensive Stress Management Guide** with evidence-based strategies from the HealthWellnessGuide.txt file. The guide includes **6 major evidence-based strategies** (exceeding your requirement of 5):

### The 6 Evidence-Based Stress Management Strategies:

1. **Deep Breathing and Progressive Muscle Relaxation** - Includes box breathing technique and systematic muscle tension-release methods
2. **Mindfulness Meditation** - Various forms including focused attention, body scan, and loving-kindness practices
3. **Regular Physical Exercise** - Aerobic, strength training, and mind-body exercises with specific guidelines
4. **Quality Sleep Optimization** - Sleep hygi

---
# 🤝 Breakout Room #2
## Advanced Features & Integration

## Task 6: Subagent Spawning

The third key element is **Subagent Spawning**. This allows a Deep Agent to delegate tasks to specialized subagents.

### Why Subagents?

1. **Context Isolation**: Each subagent has its own context window, preventing bloat
2. **Specialization**: Different subagents can have different tools/prompts
3. **Parallelism**: Multiple subagents can work simultaneously
4. **Cost Optimization**: Use cheaper models for simpler subtasks

### How Subagents Work

```
Main Agent
    ├── task("Research sleep science", model="gpt-4o-mini")
    │       └── Returns: Summary of findings
    │
    ├── task("Analyze user's sleep data", tools=[analyze_tool])
    │       └── Returns: Analysis results
    │
    └── task("Write recommendations", system_prompt="Be concise")
            └── Returns: Final recommendations
```

Key benefit: The main agent only receives **summaries**, not all the intermediate context!

In [21]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from langchain.chat_models import init_chat_model

# Define specialized subagent configurations
# Note: Subagents inherit the backend from the parent agent
research_subagent = {
    "name": "research-agent",
    "description": "Use this agent to research wellness topics in depth. It can read documents and synthesize information.",
    "system_prompt": """You are a wellness research specialist. Your job is to:
1. Find relevant information in provided documents
2. Synthesize findings into clear summaries
3. Cite sources when possible

Be thorough but concise. Focus on evidence-based information.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",  # Cheaper model for research
}

writing_subagent = {
    "name": "writing-agent",
    "description": "Use this agent to create well-structured documents, plans, and guides.",
    "system_prompt": """You are a wellness content writer. Your job is to:
1. Take research findings and turn them into clear, actionable content
2. Structure information for easy understanding
3. Use formatting (headers, bullets, etc.) effectively

Write in a supportive, encouraging tone.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "anthropic:claude-sonnet-4-20250514",
}

print("Subagent configurations defined!")

Subagent configurations defined!


In [22]:
# Create a coordinator agent that can spawn subagents
coordinator_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[write_todos, update_todo, list_todos],
    backend=filesystem_backend,  # Use the same backend - subagents inherit it
    subagents=[research_subagent, writing_subagent],
    system_prompt="""You are a Wellness Project Coordinator. Your role is to:
1. Break down complex wellness requests into subtasks
2. Delegate research to the research-agent
3. Delegate content creation to the writing-agent
4. Coordinate the overall workflow using todos

Use subagents for specialized work rather than doing everything yourself.
This keeps the work organized and the results high-quality."""
)

print("Coordinator agent created with subagent capabilities!")

Coordinator agent created with subagent capabilities!


In [23]:
# Reset for demo
TODO_STORE.clear()

# Test the coordinator with a complex task
result = coordinator_agent.invoke({
    "messages": [{
        "role": "user",
        "content": """Create a comprehensive morning routine guide for better energy.
        
The guide should:
1. Research the science behind morning routines
2. Include practical steps for exercise, nutrition, and mindset
3. Be saved as a well-formatted markdown file"""
    }]
})

print("Coordinator response:")
print(result["messages"][-1].content)

Coordinator response:
Perfect! I've successfully created your comprehensive morning routine guide for better energy. Here's what I've delivered:

## 🎯 **Complete Project Summary**

**✅ All Tasks Completed:**
1. **Researched the science** behind morning routines and their impact on energy
2. **Researched practical components** for exercise, nutrition, and mindset
3. **Created a comprehensive guide** that synthesizes all research into actionable steps
4. **Saved as a well-formatted markdown file** at `/comprehensive_morning_routine_guide.md`

## 📋 **Your Guide Includes:**

### **📚 Scientific Foundation:**
- Circadian rhythm research and why mornings matter
- Neuroplasticity advantages for habit formation
- Specific research on exercise timing, nutrition, and mindset practices
- How morning routines affect cortisol, metabolism, and cognitive function

### **🛠️ Practical Implementation:**
- **6-step core routine template** that anyone can follow
- **4 different variations**: Beginner (30-4

In [24]:
# Check the results
print("Final todo status:")
print(list_todos.invoke({}))

print("\nGenerated files in workspace:")
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

Final todo status:
✅ [todo_1] Research the science behind morning routines (completed)
✅ [todo_3] Research practical morning routine components (completed)
✅ [todo_5] Create comprehensive morning routine guide (completed)
✅ [todo_7] Save guide as formatted markdown file (completed)

Generated files in workspace:
  [FILE] comprehensive_morning_routine_guide.md (13630 bytes)
  [FILE] morning_routine_guide.md (16755 bytes)
  [FILE] personalized_sleep_improvement_plan.md (6247 bytes)
  [DIR] research/
  [FILE] sleep_improvement_research_summary.md (7839 bytes)


## Task 7: Long-term Memory Integration

The fourth key element is **Long-term Memory**. Deep Agents integrate with LangGraph's Store for persistent memory across sessions.

### Memory Types in Deep Agents

| Type | Scope | Use Case |
|------|-------|----------|
| **Thread Memory** | Single conversation | Current session context |
| **User Memory** | Across threads, per user | User preferences, history |
| **Shared Memory** | Across all users | Common knowledge, learned patterns |

### Integration with LangGraph Store

Deep Agents can use the same `InMemoryStore` (or `PostgresStore`) we learned in Session 6:

In [25]:
from langgraph.store.memory import InMemoryStore

# Create a memory store
memory_store = InMemoryStore()

# Store user profile
user_id = "user_alex"
profile_namespace = (user_id, "profile")

memory_store.put(profile_namespace, "name", {"value": "Alex"})
memory_store.put(profile_namespace, "goals", {
    "primary": "improve energy levels",
    "secondary": "better sleep"
})
memory_store.put(profile_namespace, "conditions", {
    "dietary": ["vegetarian"],
    "medical": ["mild anxiety"]
})
memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "detailed"
})

print(f"Stored profile for {user_id}")

# Retrieve and display
for item in memory_store.search(profile_namespace):
    print(f"  {item.key}: {item.value}")

Stored profile for user_alex
  name: {'value': 'Alex'}
  goals: {'primary': 'improve energy levels', 'secondary': 'better sleep'}
  conditions: {'dietary': ['vegetarian'], 'medical': ['mild anxiety']}
  preferences: {'exercise_time': 'morning', 'communication_style': 'detailed'}


In [26]:
# Create memory-aware tools
from langgraph.store.base import BaseStore

@tool
def get_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(memory_store.search(namespace))
    
    if not items:
        return f"No profile found for {user_id}"
    
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)

@tool
def save_user_preference(user_id: str, key: str, value: str) -> str:
    """Save a user preference to long-term memory.
    
    Args:
        user_id: The user's unique identifier
        key: The preference key
        value: The preference value
    
    Returns:
        Confirmation message
    """
    namespace = (user_id, "preferences")
    memory_store.put(namespace, key, {"value": value})
    return f"Saved preference '{key}' for {user_id}"

print("Memory tools defined!")

Memory tools defined!


In [27]:
# Create a memory-enhanced agent
memory_tools = [
    get_user_profile,
    save_user_preference,
    write_todos,
    update_todo,
    list_todos,
]

memory_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=memory_tools,
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a Personal Wellness Assistant with long-term memory.

At the start of each conversation:
1. Check the user's profile to understand their goals and conditions
2. Personalize all advice based on their profile
3. Save any new preferences they mention

Always reference stored information to show you remember the user."""
)

print("Memory-enhanced agent created!")

Memory-enhanced agent created!


In [28]:
# Test the memory agent
TODO_STORE.clear()

result = memory_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi! My user_id is user_alex. What exercise routine would you recommend for me?"
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Hello Alex! Great to see you again. Based on your profile, I can see your primary goal is to improve energy levels and get better sleep, and you prefer morning workouts. Given your mild anxiety, I'll recommend a routine that combines energy-boosting exercises with stress-reducing activities.

Here's a personalized exercise routine for you:

## **Morning Energy & Sleep Enhancement Routine**

### **Monday, Wednesday, Friday - Energizing Strength & Cardio (30-40 minutes)**
- **Warm-up (5 minutes)**: Light stretching and arm circles
- **Strength Circuit (20 minutes)**: 
  - Bodyweight squats: 3 sets of 12-15
  - Push-ups (modified if needed): 3 sets of 8-12
  - Plank holds: 3 sets of 30-45 seconds
  - Lunges: 3 sets of 10 each leg
- **Cardio Boost (10 minutes)**: Brisk walking or light jogging
- **Cool-down (5 minutes)**: Deep breathing and gentle stretches

### **Tuesday, Thursday - Anxiety-Reducing Yoga/Mindful Movement (25-30 minutes)**
- **Morning Flow Sequence**: Sun s

## Task 8: Skills - On-Demand Capabilities

**Skills** are a powerful feature for progressive capability disclosure. Instead of loading all tools upfront, agents can load specialized capabilities on demand.

### Why Skills?

1. **Context Efficiency**: Don't waste context on unused tool descriptions
2. **Specialization**: Skills can include detailed instructions for specific tasks
3. **Modularity**: Easy to add/remove capabilities
4. **Discoverability**: Agent can browse available skills

### SKILL.md Format

Skills are defined in markdown files with YAML frontmatter:

```markdown
---
name: skill-name
description: What this skill does
version: 1.0.0
tools:
  - tool1
  - tool2
---

# Skill Instructions

Detailed steps for how to use this skill...
```

In [29]:
# Let's look at the skills we created
skills_dir = Path("skills")

print("Available skills:")
for skill_dir in skills_dir.iterdir():
    if skill_dir.is_dir():
        skill_file = skill_dir / "SKILL.md"
        if skill_file.exists():
            content = skill_file.read_text()
            # Extract name and description from frontmatter
            lines = content.split("\n")
            name = ""
            desc = ""
            for line in lines:
                if line.startswith("name:"):
                    name = line.split(":", 1)[1].strip()
                if line.startswith("description:"):
                    desc = line.split(":", 1)[1].strip()
            print(f"  - {name}: {desc}")

Available skills:
  - meal-planning: Create personalized meal plans based on dietary needs and preferences
  - wellness-assessment: Assess user wellness goals and create personalized recommendations


In [30]:
# Read the wellness-assessment skill
skill_content = Path("skills/wellness-assessment/SKILL.md").read_text()
print(skill_content)

---
name: wellness-assessment
description: Assess user wellness goals and create personalized recommendations
version: 1.0.0
tools:
  - read_file
  - write_file
---

# Wellness Assessment Skill

You are conducting a comprehensive wellness assessment. Follow these steps:

## Step 1: Gather Information
Ask the user about:
- Current health goals (weight, fitness, stress, sleep)
- Any medical conditions or limitations
- Current exercise routine (or lack thereof)
- Dietary preferences and restrictions
- Sleep patterns and quality
- Stress levels and sources

## Step 2: Analyze Responses
Review the user's answers and identify:
- Primary wellness priority
- Secondary goals
- Potential barriers to success
- Existing healthy habits to build on

## Step 3: Create Assessment Report
Write a wellness assessment report to `workspace/wellness_assessment.md` containing:
- Summary of current wellness state
- Identified strengths
- Areas for improvement
- Recommended focus areas (prioritized)
- Suggeste

In [31]:
# Create a skill-aware tool
@tool
def load_skill(skill_name: str) -> str:
    """Load a skill's instructions for a specialized task.
    
    Available skills:
    - wellness-assessment: Assess user wellness and create recommendations
    - meal-planning: Create personalized meal plans
    
    Args:
        skill_name: Name of the skill to load
    
    Returns:
        Skill instructions
    """
    skill_path = Path(f"skills/{skill_name}/SKILL.md")
    if not skill_path.exists():
        available = [d.name for d in Path("skills").iterdir() if d.is_dir()]
        return f"Skill '{skill_name}' not found. Available: {', '.join(available)}"
    
    return skill_path.read_text()

print("Skill loader defined!")

Skill loader defined!


In [32]:
# Create an agent that can load and use skills
skill_agent = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        load_skill,
        write_todos,
        update_todo,
        list_todos,
    ],
    backend=filesystem_backend,  # Use workspace for file operations
    system_prompt="""You are a wellness assistant with access to specialized skills.

When a user asks for something that matches a skill:
1. Load the appropriate skill using load_skill()
2. Follow the skill's instructions carefully
3. Save outputs as specified in the skill

Available skills:
- wellness-assessment: For comprehensive wellness evaluations
- meal-planning: For creating personalized meal plans

If no skill matches, use your general wellness knowledge."""
)

print("Skill-aware agent created!")

Skill-aware agent created!


In [33]:
# Test with a skill-appropriate request
TODO_STORE.clear()

result = skill_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "I'd like a wellness assessment. I'm a 35-year-old office worker who sits most of the day, has trouble sleeping, and wants to lose 15 pounds. I'm vegetarian and have no major health conditions."
    }]
})

print("Agent response:")
print(result["messages"][-1].content)

Agent response:
Now let me provide you with specific, actionable recommendations based on your assessment:

## Your Wellness Assessment Results

Based on your information, I've identified that your **sedentary lifestyle is the primary factor** affecting both your sleep quality and weight management goals. The good news is that you have a strong foundation with no major health conditions and already making conscious food choices as a vegetarian.

## Immediate Action Items (Start Today)

1. **Set hourly movement reminders** - Use your phone/computer to alert you every hour to stand and move for 2-3 minutes
2. **Establish a consistent bedtime** - Choose a specific time (e.g., 10:30 PM) and stick to it for the next week
3. **Add a 10-minute evening walk** - This will help with both activity and sleep preparation

## Short-Term Goals (1-2 Weeks)

1. **Implement desk exercises** - 5-minute stretching/movement routine every 2 hours during work
2. **Create a wind-down routine** - 30 minutes be

## Task 9: Using deepagents-cli

The `deepagents-cli` provides an interactive terminal interface for working with Deep Agents.

### Installation

```bash
uv pip install deepagents-cli
# or
pip install deepagents-cli
```

### Key Features

| Feature | Description |
|---------|-------------|
| **Interactive Sessions** | Chat with your agent in the terminal |
| **Conversation Resume** | Pick up where you left off |
| **Human-in-the-Loop** | Approve or reject agent actions |
| **File System Access** | Agent can read/write to your filesystem |
| **Remote Sandboxing** | Run in isolated Docker containers |

### Basic Usage

```bash
# Start an interactive session
deepagents

# Resume a previous conversation
deepagents --resume

# Use a specific model
deepagents --model openai:gpt-4o

# Enable human-in-the-loop approval
deepagents --approval-mode full
```

### Example Session

```
$ deepagents

Welcome to Deep Agents CLI!

You: Create a 7-day meal plan for a vegetarian athlete

Agent: I'll create a comprehensive meal plan for you. Let me:
1. Research vegetarian athlete nutrition needs
2. Design balanced daily menus
3. Save the plan to a file

[Agent uses tools...]

Agent: I've created your meal plan! You can find it at:
workspace/vegetarian_athlete_meal_plan.md

You: /exit
```

In [34]:
# Check if CLI is installed
import subprocess

try:
    result = subprocess.run(["deepagents", "--version"], capture_output=True, text=True)
    print(f"deepagents-cli version: {result.stdout.strip()}")
except FileNotFoundError:
    print("deepagents-cli not installed. Install with:")
    print("  uv pip install deepagents-cli")
    print("  # or")
    print("  pip install deepagents-cli")

deepagents-cli version: deepagents 0.0.19


### Try It Yourself!

After installing the CLI, try these commands in your terminal:

```bash
# Basic interactive session
deepagents

# With a specific working directory
deepagents --workdir ./workspace

# See all options
deepagents --help
```

Sample prompts to try:
1. "Create a weekly workout plan and save it to a file"
2. "Research the health benefits of meditation and summarize in a report"
3. "Analyze my current diet and suggest improvements" (then provide details)

## Task 10: Building a Complete Deep Agent System

Now let's bring together all four elements to build a comprehensive "Wellness Coach" system:

1. **Planning**: Track multi-week wellness programs
2. **Context Management**: Store session notes and progress
3. **Subagent Spawning**: Delegate to specialists (exercise, nutrition, mindfulness)
4. **Long-term Memory**: Remember user preferences and history

In [35]:
# Define specialized wellness subagents
# Subagents inherit the backend from the parent, so they use the same workspace
exercise_specialist = {
    "name": "exercise-specialist",
    "description": "Expert in exercise science, workout programming, and physical fitness. Use for exercise-related questions and plan creation.",
    "system_prompt": """You are an exercise specialist with expertise in:
- Workout programming for different fitness levels
- Exercise form and safety
- Progressive overload principles
- Recovery and injury prevention

Always consider the user's fitness level and any physical limitations.
Provide clear, actionable exercise instructions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

nutrition_specialist = {
    "name": "nutrition-specialist",
    "description": "Expert in nutrition science, meal planning, and dietary optimization. Use for food-related questions and meal plans.",
    "system_prompt": """You are a nutrition specialist with expertise in:
- Macro and micronutrient balance
- Meal planning and preparation
- Dietary restrictions and alternatives
- Nutrition timing for performance

Always respect dietary restrictions and preferences.
Focus on practical, achievable meal suggestions.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

mindfulness_specialist = {
    "name": "mindfulness-specialist",
    "description": "Expert in stress management, sleep optimization, and mental wellness. Use for stress, sleep, and mental health questions.",
    "system_prompt": """You are a mindfulness and mental wellness specialist with expertise in:
- Stress reduction techniques
- Sleep hygiene and optimization
- Meditation and breathing exercises
- Work-life balance strategies

Be supportive and non-judgmental.
Provide practical techniques that can be implemented immediately.""",
    "tools": [],  # Uses built-in file tools from backend
    "model": "openai:gpt-4o-mini",
}

print("Specialist subagents defined!")

Specialist subagents defined!


In [36]:
# Create the Wellness Coach coordinator
wellness_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        # Planning
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_user_profile,
        save_user_preference,
        # Skills
        load_skill,
    ],
    backend=filesystem_backend,  # All file ops go to workspace
    subagents=[exercise_specialist, nutrition_specialist, mindfulness_specialist],
    system_prompt="""You are a Personal Wellness Coach that coordinates comprehensive wellness programs.

## Your Role
- Understand each user's unique goals, constraints, and preferences
- Create personalized, multi-week wellness programs
- Coordinate between exercise, nutrition, and mindfulness specialists
- Track progress and adapt recommendations

## Workflow
1. **Initial Assessment**: Get user profile and understand their situation
2. **Planning**: Create a todo list for the program components
3. **Delegation**: Use specialists for domain-specific content:
   - exercise-specialist: Workout plans and fitness guidance
   - nutrition-specialist: Meal plans and dietary advice
   - mindfulness-specialist: Stress and sleep optimization
4. **Integration**: Combine specialist outputs into a cohesive program
5. **Documentation**: Save all plans and recommendations to files

## Important
- Always check user profile first for context
- Respect any medical conditions or dietary restrictions
- Provide clear, actionable recommendations
- Save progress to files so users can reference later"""
)

print("Wellness Coach created with all 4 Deep Agent elements!")

Wellness Coach created with all 4 Deep Agent elements!


In [37]:
# Test the complete system
TODO_STORE.clear()

result = wellness_coach.invoke({
    "messages": [{
        "role": "user",
        "content": """Hi! My user_id is user_alex. I'd like you to create a 2-week wellness program for me.

I want to focus on:
1. Building a consistent exercise routine (I can exercise 3x per week for 30 mins)
2. Improving my diet (remember I'm vegetarian)
3. Better managing my work stress and improving my sleep

Please create comprehensive plans for each area and save them as separate files I can reference."""
    }]
})

print("Wellness Coach response:")
print(result["messages"][-1].content)

Wellness Coach response:
Perfect! I've created your comprehensive 2-week wellness program with all the components you requested. Here's what you now have:

## 🎯 **Your Complete Wellness Program Files:**

1. **📋 Master Program**: `/alex_complete_2week_wellness_program.md` - Your integrated daily schedule
2. **💪 Exercise Plan**: `/alex_exercise_plan_2weeks.md` - 6 detailed workout sessions (30 mins each, 3x/week)
3. **🥗 Vegetarian Meal Plan**: `/alex_vegetarian_meal_plan_2weeks.md` - Complete meals focused on energy & anxiety support
4. **🧘 Stress & Sleep Optimization**: `/alex_stress_sleep_optimization_2weeks.md` - Daily routines for work stress and sleep improvement

## ✨ **Program Highlights:**

**Exercise**: Morning workouts perfectly timed for your preference, progressing from basic bodyweight exercises to more challenging combinations

**Nutrition**: Vegetarian meals rich in energy-boosting nutrients, magnesium for anxiety management, and foods that support better sleep

**Stress M

In [36]:
# Review what was created
print("=" * 60)
print("FINAL TODO STATUS")
print("=" * 60)
print(list_todos.invoke({}))

print("\n" + "=" * 60)
print("GENERATED FILES")
print("=" * 60)
for f in sorted(WORKSPACE.iterdir()):
    if f.is_file():
        print(f"  [FILE] {f.name} ({f.stat().st_size} bytes)")
    elif f.is_dir():
        print(f"  [DIR] {f.name}/")

FINAL TODO STATUS
✅ [todo_1] Assess user profile and preferences (completed)
✅ [todo_3] Create exercise plan (completed)
✅ [todo_5] Create nutrition plan (completed)
✅ [todo_7] Create stress & sleep optimization plan (completed)
✅ [todo_9] Integrate all plans into cohesive program (completed)
✅ [todo_11] Save individual plans to files (completed)
✅ [todo_13] Save complete program overview (completed)

GENERATED FILES
  [FILE] .gitkeep (0 bytes)
  [FILE] Alex_2_Week_Exercise_Program.txt (4330 bytes)
  [FILE] Alex_Complete_2Week_Wellness_Program.md (4757 bytes)
  [FILE] Alex_Exercise_Plan.md (3864 bytes)
  [FILE] Alex_Mindfulness_Plan.md (6396 bytes)
  [FILE] Alex_Nutrition_Plan.md (5482 bytes)
  [FILE] complete_14_day_vegetarian_meal_plan.txt (5045 bytes)
  [FILE] comprehensive_morning_routine_guide.md (25117 bytes)
  [FILE] meal_plan_nutritional_highlights.txt (468 bytes)
  [FILE] meal_plan_week_1.txt (1765 bytes)
  [FILE] meal_plan_week_2.txt (1598 bytes)
  [FILE] meal_prep_tips.txt (

In [37]:
# Read one of the generated files
files = list(WORKSPACE.glob("*.md"))
if files:
    print(f"\nContents of {files[0].name}:")
    print("=" * 60)
    print(files[0].read_text()[:2000] + "..." if len(files[0].read_text()) > 2000 else files[0].read_text())


Contents of morning_routine_guide.md:
# The Ultimate Morning Routine Guide: Energize Your Day for Peak Performance

## Table of Contents
- [Introduction: Why Your Morning Sets the Tone](#introduction-why-your-morning-sets-the-tone)
- [The Science-Backed Foundation](#the-science-backed-foundation)
- [Exercise: Wake Up Your Body](#exercise-wake-up-your-body)
- [Nutrition: Fuel Your Success](#nutrition-fuel-your-success)
- [Mindset Practices: Train Your Mental Muscle](#mindset-practices-train-your-mental-muscle)
- [Implementation Strategies: Building Lasting Habits](#implementation-strategies-building-lasting-habits)
- [Common Mistakes to Avoid](#common-mistakes-to-avoid)
- [Customization for Your Lifestyle](#customization-for-your-lifestyle)
- [Sample Morning Routine Templates](#sample-morning-routine-templates)
- [Your Next Steps](#your-next-steps)

---

## Introduction: Why Your Morning Sets the Tone

Picture this: You wake up feeling refreshed, energized, and ready to tackle whatever

---
## ❓ Question #3:

What are the key considerations when designing **subagent configurations**?

Consider:
- When should subagents share tools vs have distinct tools?
- How do you decide which model to use for each subagent?
- What's the right granularity for subagent specialization?

##### Answer:
- A subagent is meant to wrap a part of the context from our deep agent when that context can be self-contained.
- If that agent requires access to specialized tools then we can organize these tools under that sub-agent. Sometimes however a more generic tool might be useful for it to complete a job (e.g., a calculator, a web search, etc.). If I were to build a deep agent system with subagents I would first start with specialized tools and if I find myself duplicating tool work then I would refactor to commonize some tools (same way we build libs in software when we find common code)
- For model selection I'd start with the cheapest/fastest model and assess its quality. Before moving to a bigger model I'd ask myself whether I've defined the sub-agent's problem well-enough because ideally I should be able to do the work with a "lighter" model.
- In terms of granularity, I'd start with an idea for a task and I'd consider a few things: 1) Can the task be completed in isolation? 2) Can the task be completed in parallel to other tasks? 3) Can it be completed with the simple LLM + memory + RAG + tools alone? If I see the task is more complex than that I may have to split it up.

## ❓ Question #4:

For a **production wellness application** using Deep Agents, what would you need to add?

Consider:
- Safety guardrails for health advice
- Persistent storage (not in-memory)
- Multi-user support and isolation
- Monitoring and observability
- Cost management with subagents

##### Answer:
A few things to consider (similar to a production web app):
- Reproducible tests/evals with test prompts
- Proper dependency injection so I can boot up my agent in development, staging, production with different configs, especially for data storage and retrieval for multi-tenancy (multiple customer tenants) and multi-region (US, EU, etc.)
- Robust guardrails by testing my prompts or a sub-agent to validate adversarially the results. Possibly having to also deal with laws in different countries where the users reside.
- Feedback collection on response quality
- Performance metrics: tokens, cost, latency etc.
- Testing with multiple user personas (different allergies, preferences, etc.)

---
## 🏗️ Activity #2: Build a Wellness Coach Agent

Build your own wellness coach that uses all 4 Deep Agent elements.

### Requirements:
1. **Planning**: Create todos for a 30-day wellness challenge
2. **Context Management**: Store daily check-in notes
3. **Subagents**: At least 2 specialized subagents
4. **Memory**: Remember user preferences across interactions

### Challenge:
Create a "30-Day Wellness Challenge" system that:
- Generates a personalized 30-day plan
- Tracks daily progress
- Adapts recommendations based on feedback
- Saves a weekly summary report

In [60]:
### YOUR CODE HERE ###

# Step 1: Create any additional tools you need. Moved those up since my
# sub-agents are using them.
coach_memory_store = InMemoryStore()

# Store the user profile in the memory store
user_id = "user_nikos"
profile_namespace = (user_id, "profile")

coach_memory_store.put(profile_namespace, "name", {"value": "Nikos"})
coach_memory_store.put(profile_namespace, "goals", {
    "primary": "reduse stress levels",
    "secondary": "lose weight"
})
coach_memory_store.put(profile_namespace, "conditions", {
    "allergies": ["pollen"],
    "medical": ["bad back"]
})
coach_memory_store.put(profile_namespace, "preferences", {
    "exercise_time": "morning",
    "communication_style": "punchy"
})

print(f"Stored profile for {user_id}")

@tool
def get_coach_user_profile(user_id: str) -> str:
    """Retrieve a user's wellness profile from long-term memory.
    
    Args:
        user_id: The user's unique identifier
    
    Returns:
        User profile as formatted text
    """
    namespace = (user_id, "profile")
    items = list(coach_memory_store.search(namespace))
    
    if not items:
        return f"No profile found for {user_id}"
    
    result = [f"Profile for {user_id}:"]
    for item in items:
        result.append(f"  {item.key}: {item.value}")
    return "\n".join(result)


@tool
def log_update_30_day_plan(user_id: str, day: int, update: str) -> str:
    """Update the 30-day plan in the memory store.
    
    Args:
        user_id: The ID of the user to update the plan for.
        day: The day of the plan to update (1-30).
        update: The one-sentence summary of the update made to the plan.
    """
    plan_namespace = (user_id, "plan")
    coach_memory_store.put(plan_namespace, f"{day:02d}", {"update": update})
    return f"Updated 30-day plan for {user_id} on day {day} with: {update}"


@tool
def log_user_checkin(user_id: str, activity: str, feedback: str, day: int) -> str:
    """Log the user's checkin to the memory store.
    
    Args:
        user_id: The ID of the user to log the feedback for.
        activity: The activity that the user did.
        feedback: The feedback to log.
        day: The day of the feedback during the challenge (1-30).
    """
    checkin_namespace = (user_id, "checkin")
    coach_memory_store.put(
        checkin_namespace, 
        f"{day:02d}", 
        {"activity": activity, "feedback": feedback}
    )
    return f"Logged checkin for {user_id} on day {day}"


@tool
def get_user_feedback_since(user_id: str, since_day: int) -> list[dict[str, str]]:
    """Get the user's feedback from the memory store.
    
    Args:
        user_id: The user_id of the user to get the feedback for.
        since_day: The day to start searching user feedback from (1-30).

    Returns:
        The user's feedback as a dictionary of day -> feedback.
    """
    feedback_namespace = (user_id, "checkin")
    items = coach_memory_store.search(feedback_namespace)
    return [{"day": int(item.key), "feedback": item.value} for item in items if int(item.key) >= since_day]

# Step 2: Define your subagent configurations
feedback_specialist = {
    "name": "feedback-specialist",
    "description": "Expert in user feedback collection and analysis. Use for feedback collection and analysis.",
    "system_prompt": """You are a user feedback analysis specialist. Your job is to:
1. Read the user's daily feedback logs.
2. Analyze the feedback and provide a clear, actionable recommendations for the next day.
""",
    "tools": [get_user_feedback_since],
    "model": "openai:gpt-4o-mini",
}

weekly_summary_specialist = {
    "name": "weekly-summary-specialist",
    "description": "Expert in weekly summary creation. Use for weekly summary creation.",
    "system_prompt": """You are a weekly summary specialist. Your job is to:
1. Read the last week's user progress logs and wellness plan for that week.
2. Create a weekly summary report with the user's progress.
""",
    "tools": [get_user_feedback_since],
    "model": "openai:gpt-4o-mini",
}

# Step 3: Build the main coordinator agent
coach_workspace_path = Path("coach_workspace").absolute()
coach_backend = FilesystemBackend(
    root_dir=str(coach_workspace_path),
    virtual_mode=True  # This is required to sandbox file operations!
)

challenge_coach = create_deep_agent(
    model=init_chat_model("anthropic:claude-sonnet-4-20250514"),
    tools=[
        write_todos,
        update_todo,
        list_todos,
        # Long-term Memory
        get_coach_user_profile,
        get_user_feedback_since,
        log_update_30_day_plan,
        # Wellness guide for research
        read_health_wellness_guide
    ],
    backend=coach_backend,
    subagents=[feedback_specialist, weekly_summary_specialist],
    system_prompt="""You are a 30-day wellness challenge coach. 
## Your role:
1. Create and save a personalized 30-day plan for the user explaining what to do each specific day.
2. Track the user's progress daily.
3. Adapt the plan based on the user's activity and feedback.
4. Save a weekly summary report every 7 days.

## Workflow:
1. Get the user's profile and understand their situation.
2. Create a todo list for the program components if the request is complicated.
3. Use the wellness guide for research exclusively. 
4. Use the feedback specialist to check if the user has checked in. 
   If they have, analyze the user's feedback and provide a clear, 
   actionable recommendations for the next day and update the 30-day plan accordingly.
5. Log what you updated in the 30-day plan every time you make an adjustment.
5. Use the weekly summary specialist to create a weekly summary report.

## Important:
- Always check the user's profile first for context and don't ask for additional information.
- Always use the wellness guide for research exclusively.
- Always log what you updated in the 30-day plan every time you make an adjustment.
- Always use the weekly summary specialist to create a weekly summary report 
  after the user has logged their checkins for the week.
"""
)
# Step 4: Test with a user creating their 30-day challenge
TODO_STORE.clear()

result = challenge_coach.invoke({
    "messages": [{
        "role": "user",
        "content": "Hi, my user_id is user_nikos. I want to create a 30-day wellness challenge. Please create a plan for me."
    }]
})

print("Challenge Coach response:")
print(result["messages"][-1].content)


Stored profile for user_nikos
Challenge Coach response:
Perfect! I've created your personalized 30-day wellness challenge plan, Nikos! 🎯

## Your Plan Highlights:

**✅ Stress-Busting Focus:**
- Daily morning breathing exercises (your favorite punchy 4-4-4 pattern)
- Progressive stress management techniques
- Evening wind-down routines

**✅ Back-Safe Workouts:**
- All exercises chosen specifically for bad back conditions
- Cat-cow stretches, bird dogs, pelvic tilts
- No twisting or high-impact moves

**✅ Morning-Focused Schedule:**
- All main activities in your preferred morning time
- 20-40 minute sessions that fit your schedule
- Quick, effective routines

**✅ Weight Loss Support:**
- Nutrition guidance focused on whole foods
- Portion control strategies
- Anti-inflammatory meal planning

**✅ Allergy Considerations:**
- Indoor exercise alternatives for high pollen days
- Flexible outdoor activities
- Pollen level awareness reminders

## How It Works:
- **Week 1:** Foundation building 

In [61]:
# Step 5: Simulate a daily check-in and adaptation

# Let's clear previous runs.
coach_memory_store.search(("user_nikos", "checkin")).clear()

log_user_checkin.invoke({
    "user_id": "user_nikos",
    "activity": "walking",
    "feedback": "I felt that after walking for one hour it was too easy.",
    "day": 1
})
log_user_checkin.invoke({
    "user_id": "user_nikos",
    "activity": "stretching",
    "feedback": "Stretching was good for my back.",
    "day": 2
})
log_user_checkin.invoke({
    "user_id": "user_nikos",
    "activity": "yoga",
    "feedback": "Yoga was good for my back.",
    "day": 3
})
log_user_checkin.invoke({
    "user_id": "user_nikos",
    "activity": "gym",
    "feedback": "I ran on the treadmill and lifted weights and I felt great. No back pain.",
    "day": 4
})
log_user_checkin.invoke({
    "user_id": "user_nikos",
    "activity": "walking",
    "feedback": "I felt that after walking for one hour it was still too easy.",
    "day": 5
})
log_user_checkin.invoke({
    "user_id": "user_nikos",
    "activity": "eating",
    "feedback": "I've been eating salads every day this week. I'm not sure I'm losing weight.",
    "day": 6
})
log_user_checkin.invoke({
    "user_id": "user_nikos",
    "activity": "gym",
    "feedback": "I've been going to the gym every day this week. I'm not sure I'm losing weight.",
    "day": 7
})
log_user_checkin.invoke({
    "user_id": "user_nikos",
    "activity": "gym",
    "feedback": "I've been going to the gym every day this week. I'm not sure I'm losing weight.",
    "day": 8
})

result = get_user_feedback_since.invoke({
    "user_id": "user_nikos",
    "since_day": 8
})

print("User feedback since day 8:")
print(result)

result =challenge_coach.invoke({
    "messages": [{
        "role": "user",
        "content": "This is user_nikos again. I just finished day 8. What's my plan for tomororw? Are you changing anything for week 2?"
    }]
})

print("Challenge Coach response:")
print(result["messages"][-1].content)


User feedback since day 8:
[{'day': 8, 'feedback': {'activity': 'gym', 'feedback': "I've been going to the gym every day this week. I'm not sure I'm losing weight."}}]
Challenge Coach response:
**Tomorrow's focus**: Give your body some recovery time while getting clarity on your nutrition. The key insight is that weight loss is about 70% diet, 30% exercise. Your gym consistency is excellent - now we need to dial in the nutrition piece!

Track everything you eat tomorrow and let me know how it goes. Your dedication is paying off with no back pain - let's get that weight moving too! 💪


---
## Summary

In this session, we explored **Deep Agents** and their four key elements:

| Element | Purpose | Implementation |
|---------|---------|----------------|
| **Planning** | Track complex tasks | `write_todos`, `update_todo`, `list_todos` |
| **Context Management** | Handle large contexts | File system tools, automatic offloading |
| **Subagent Spawning** | Delegate to specialists | `task` tool with custom configs |
| **Long-term Memory** | Remember across sessions | LangGraph Store integration |

### Key Takeaways:

1. **Deep Agents handle complexity** - Unlike simple tool loops, they can manage long-horizon, multi-step tasks
2. **Planning is context engineering** - Todo lists and files aren't just organization—they're extended memory
3. **Subagents prevent context bloat** - Delegation keeps the main agent focused and efficient
4. **Skills enable progressive disclosure** - Load capabilities on-demand instead of upfront
5. **The CLI makes interaction natural** - Interactive sessions with conversation resume

### Deep Agents vs Traditional Agents

| Aspect | Traditional Agent | Deep Agent |
|--------|-------------------|------------|
| Task complexity | Simple, single-step | Complex, multi-step |
| Context management | All in conversation | Files + summaries |
| Delegation | None | Subagent spawning |
| Memory | Within thread | Across sessions |
| Planning | Implicit | Explicit (todos) |

### When to Use Deep Agents

**Use Deep Agents when:**
- Tasks require multiple steps or phases
- Context would overflow in a simple loop
- Specialization would improve quality
- Users need to resume sessions
- Long-term memory is valuable

**Use Simple Agents when:**
- Tasks are straightforward Q&A
- Single tool call suffices
- Context fits easily
- No need for persistence

### Further Reading

- [Deep Agents Documentation](https://docs.langchain.com/oss/python/deepagents/overview)
- [Deep Agents GitHub](https://github.com/langchain-ai/deepagents)
- [Context Management Blog Post](https://www.blog.langchain.com/context-management-for-deepagents/)
- [Building Multi-Agent Applications](https://www.blog.langchain.com/building-multi-agent-applications-with-deep-agents/)
- [LangGraph Memory Concepts](https://langchain-ai.github.io/langgraph/concepts/memory/)